In [1]:
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
cwd = Path.cwd()
import LEAD as lead
from tqdm.notebook import tqdm


# Important variables
SNRs = np.array([-np.inf,-13,-11,-9,-7,-5,-3])
SubIDs = ['01','02','03','05','06','07','08','09','11','12','13','14','15','17','19','20','22','23','24','25']
colormap = {0: (0, 0, 0), 1: (0, 0.25, 1), 2: (0, 0.9375, 1), 3: (0, 0.91, 0.1), 4: (1, 0.6, 0), 5: (1, 0, 0), 6: (0.8, 0, 0)}

In [ ]:
from sklearn.model_selection import KFold
import os, json

# ======================================================
# PARAMETERS
# ======================================================
n_categories = 6
categories = list(range(n_categories))
n_trials = 150
n_timepoints = 250
dt = 1.0

# Input definition (same for all categories & trials)
one_input = np.concatenate((np.zeros(75), np.ones(25), np.zeros(150)))
input_series = {cat: np.stack([one_input for _ in range(n_trials)]) for cat in categories}

# ======================================================
# SYNTHETIC DATA GENERATION
# ======================================================
def w_fun(cat, regul=2, scale=7.5):
    return (np.exp(cat/regul)-1)/(scale*np.exp(5/regul))

def g_fun(cat, root=6, scale=65):
    return -cat*(cat-root)/scale

def generate_data(model_class, params):
    model = model_class(**params)
    state_series = model.measure_simulations(input_series=input_series)
    return state_series, input_series


# Ground-truth parameters
def random_params(model_type='Linear'): 
    params = dict(tau=np.random.uniform(3,20), process_noise=np.random.uniform(0.1,1), measure_noise=np.random.uniform(0,1), 
              w0=0, w1=np.random.uniform(0,0.05), w2=np.random.uniform(0.05,0.1), w3=np.random.uniform(0.1,0.2), w4=np.random.uniform(0.2,0.3), w5=np.random.uniform(0.3,0.4))
    if model_type == 'NonLinear1':
        params['gain'] = np.random.uniform(0.1, 0.5)
        params['threshold'] = np.random.uniform(0, 1.5)
        params['sharpness'] = 5
    elif model_type == 'GainModulation':
        params['threshold'] = np.random.uniform(0, 1.5)
        params['sharpness'] = 5
        w0 = np.random.uniform(0,0.1)
        regul, scale_w = np.random.uniform(1, 3), np.random.uniform(3, 15)
        root, scale_g = np.random.uniform(5, 10), np.random.uniform(55, 65)
        for cat in range(6):
            params[f'g{cat}'] = g_fun(cat, root=root, scale=scale_g)
            params[f'w{cat}'] = w0 + w_fun(cat, regul=regul, scale=scale_w)
    return params


# ======================================================
# FITTING AND CROSS-VALIDATION
# ======================================================
def crossval_loglik(state_series, input_series):
    """Compute mean test log-likelihood via 5-fold CV."""
    kf = KFold(n_splits=5, shuffle=True, random_state=0)
    trial_indices = np.arange(n_trials)
    test_lls_linear, test_lls_nonlinear, test_lls_gainmodul = [], [], []

    for train_idx, test_idx in kf.split(trial_indices):
        # Split data
        state_train, input_train = {}, {}
        state_test, input_test = {}, {}
        for cat in categories:
            state_train[cat] = state_series[cat][train_idx]
            input_train[cat] = input_series[cat][train_idx]
            state_test[cat] = state_series[cat][test_idx]
            input_test[cat] = input_series[cat][test_idx]

        # Re-organize to gather baseline windows together
        no_stim_sliced = np.concatenate([state_train[0][:, i*50:(i+1)*50] for i in range(5)])
        pre_stim_sliced = np.concatenate([state_train[cat][:, :50] for cat in range(1, n_categories)])
        state_train[0] = np.concatenate((no_stim_sliced, pre_stim_sliced))
        input_train[0] = np.zeros_like(state_train[0])


        # ---- Fit StratifiedLinear ----
        linear = lead.fitting_tools.clever_fit_linear(state_train = state_train, input_train = input_train, n_loops = 2, input_start_index=75, input_stop_index=100)
    
        # ---- Fit StratifiedGainModul  ----
        gainmodul = lead.fitting_tools.clever_fit_gainmodul(linear_prefitted = linear, state_train = state_train, input_train = input_train, n_loops = 2, input_start_index=75, input_stop_index=100)
    
        # ---- Fit StratifiedNonLinear1  ----
        nonlinear = lead.fitting_tools.clever_fit_nonlinear1(linear_prefitted = linear, state_train = state_train, input_train = input_train, n_loops = 2, input_start_index=75, input_stop_index=100)
        
        
        # ----  Isolate segments where stimulation is supposed constant ----
        state_test_constant_stim = {cat: state_test[cat][:,75:100] for cat in range(1, n_categories)}
        state_test_constant_stim[0] = np.concatenate((np.concatenate([state_test[0][:, i*50:(i+1)*50] for i in range(5)]), np.concatenate([state_test[cat][:, :50] for cat in range(1, n_categories)])))
        input_test_constant_stim = {cat: input_test[cat][:,75:100] for cat in range(1, n_categories)}
        input_test_constant_stim[0] = np.zeros_like(state_test_constant_stim[0])
        
        # ----  Evaluate on test ----
        ll_linear = linear.loglikelihood(state_test_constant_stim, input_test_constant_stim)
        test_lls_linear.append(ll_linear)
        ll_nonlinear = nonlinear.loglikelihood(state_test_constant_stim, input_test_constant_stim)
        test_lls_nonlinear.append(ll_nonlinear)
        ll_gainmodul = gainmodul.loglikelihood(state_test_constant_stim, input_test_constant_stim)
        test_lls_gainmodul.append(ll_gainmodul)

    return np.array(test_lls_linear), np.array(test_lls_nonlinear),  np.array(test_lls_gainmodul)


# ======================================================
# MODEL RECOVERY WITH CV & CV+BIC
# ======================================================
def compute_bic(ll, k, n):
    return -2 * ll + k * np.log(n)

def run_recovery(n_datasets=10, checkpoint_path="ModelRecovery/checkpoints.json", save=True):
    os.makedirs(os.path.dirname(checkpoint_path), exist_ok=True)

    if os.path.exists(checkpoint_path):
        with open(checkpoint_path, "r") as f:
            checkpoint = json.load(f)
        results_cv = checkpoint["results_cv"]
        detailed = checkpoint["detailed"]
        true_params = checkpoint["true_params"]
        linear_params = checkpoint["linear_params"]
        nonlinear_params = checkpoint["nonlinear1_params"]
        gainmodul_params = checkpoint["gainmodul_params"]
        done = checkpoint["done"]
        print(f"Resuming from checkpoint: {done} datasets already completed.")
    else:
        results_cv, results_bic, detailed, true_params, linear_params, nonlinear_params, gainmodul_params, done = [], [], [], [], [], [], [], 0

    for i in tqdm(range(done, n_datasets), initial=done, total=n_datasets):

        # Pick one type of model
        options = [("Linear", lead.model.StratifiedLinear), ("NonLinear1", lead.model.StratifiedNonLinear1), ("GainModulation", lead.model.StratifiedGainModulation)]
        true_model_name, true_class = options[i%3]

        # Generate parameters and data
        true_params_this_dataset = random_params(model_type=true_model_name)
        state_series, _ = generate_data(true_class, true_params_this_dataset)

        # ---- Save true params ----
        true_params.append(true_params_this_dataset)

        # Arrays of 5 folds each
        lls_lin, lls_nonlin, lls_gainmod = crossval_loglik(state_series, input_series)
        ll_lin_mean = float(np.mean(lls_lin))
        ll_nonlin_mean = float(np.mean(lls_nonlin))
        ll_gainmod_mean = float(np.mean(lls_gainmod))

        # ---- CV criterion ----
        candidates = ['Linear', 'NonLinear1', 'GainModulation']
        best_cv = candidates[np.argmax([ll_lin_mean, ll_nonlin_mean, ll_gainmod_mean])]
        results_cv.append((true_model_name, best_cv))
        print(f'{true_model_name} --> {best_cv}')
        

        ################### Fit on whole dataset to get final parameters ############################

        state_train, input_train = state_series.copy(), input_series.copy()

        # Re-organize to gather baseline windows together
        no_stim_sliced = np.concatenate([state_train[0][:, i*50:(i+1)*50] for i in range(5)])
        pre_stim_sliced = np.concatenate([state_train[cat][:, :50] for cat in range(1, n_categories)])
        state_train[0] = np.concatenate((no_stim_sliced, pre_stim_sliced))
        input_train[0] = np.zeros_like(state_train[0])

        # ---- Fit StratifiedLinear on whole dataset  ----
        linear = lead.fitting_tools.clever_fit_linear(state_train = state_train, input_train = input_train, n_loops = 2, input_start_index=75, input_stop_index=100)
        linear_params.append(linear.get_params())

        # ---- Fit StratifiedGainModul on whole dataset  ----
        gainmodul = lead.fitting_tools.clever_fit_gainmodul(linear_prefitted = linear, state_train = state_train, input_train = input_train, n_loops = 2, input_start_index=75, input_stop_index=100)
        gainmodul_params.append(gainmodul.get_params())

        # ---- Fit StratifiedNonLinear1 on whole dataset  ----
        nonlinear = lead.fitting_tools.clever_fit_nonlinear1(linear_prefitted = linear, state_train = state_train, input_train = input_train, n_loops = 2, input_start_index=75, input_stop_index=100)
        nonlinear_params.append(nonlinear.get_params())



        ####################### Save all da shit ############################

        # ---- Save detailed info ----
        detailed.append({
            "dataset_index": i,
            "true_model": true_model_name,
            "ll_linear_folds": lls_lin.tolist(),
            "ll_nonlinear_folds": lls_nonlin.tolist(),
            "ll_gainmod_folds": lls_gainmod.tolist(),
            "ll_linear_mean": ll_lin_mean,
            "ll_nonlinear_mean": ll_nonlin_mean,
            "ll_gainmod_mean": ll_gainmod_mean,
            "best_cv": best_cv,
        })

        # ---- Save checkpoint ----
        checkpoint = {
            "results_cv": results_cv,
            "detailed": detailed,
            "true_params": true_params,
            "linear_params": linear_params,
            "nonlinear1_params": nonlinear_params,
            "gainmodul_params": gainmodul_params,
            "done": i + 1
        }
        if save:
            with open(checkpoint_path, "w") as f:
                json.dump(checkpoint, f, indent=2)

    print("✅ All datasets processed and checkpoint saved.")
    return results_cv, detailed


# ======================================================
# CONFUSION MATRIX PLOT
# ======================================================
def plot_confusion(results, title):
    labels = ["Linear", "NonLinear1", "GainModulation"]
    mat = np.zeros((3, 3))
    for true, pred in results:
        i = labels.index(true)
        j = labels.index(pred)
        mat[i, j] += 1
    mat /= mat.sum(axis=1, keepdims=True)

    ax = plt.subplot()
    im = ax.imshow(mat, cmap="Purples", vmin=0, vmax=1)
    for i in range(3):
        for j in range(3):
            ax.text(j, i, f"{mat[i,j]:.3f}", ha="center", va="center", color="black")
    ax.set_xticks([0, 1, 2]); ax.set_xticklabels(labels)
    ax.set_yticks([0, 1, 2]); ax.set_yticklabels(labels)
    ax.set_xlabel("Predicted model")
    ax.set_ylabel("True model")
    ax.set_title(title)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    return ax


# ======================================================
# MAIN
# ======================================================

checkpoint_code = 'v2'

results_cv, detailed = run_recovery(n_datasets=200, checkpoint_path=f"ModelRecovery/checkpoints_{checkpoint_code}.json", save=True)

plt.figure()
plot_confusion(results_cv, "Model recovery v2 (5CV)")
plt.savefig(f'ModelRecovery/Linear_vs_NonLinear1_vs_GainModulation_{checkpoint_code}.png')
plt.show()

Resuming from checkpoint: 4 datasets already completed.


  2%|2         | 4/200 [00:00<?, ?it/s]